In [1]:
import sys
!{sys.executable} -m pip install rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 49.9 MB/s eta 0:00:00


In [2]:
from rdkit import Chem
print("RDKit works:", Chem.rdBase.rdkitVersion)

RDKit works: 2025.09.6


In [18]:
import os
import shutil

# Remove the blocked directory
shutil.rmtree('/content/drive', ignore_errors=True)

# Now mount fresh
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [32]:
import os

# Check your dataset is there
dataset_path = '/content/drive/MyDrive/mini_dataset'
print(os.listdir(dataset_path))

['1a30', '1bcu', '1e66', '1f8b', '1f8c', '1f8d', '1gpk', '1h23', '1hfs', '1hnn', '1igj', '1jyq', '1kel', '1lbk', '1lol', '1loq', '1lor', '1mq6', '1n1m', '1n2v', '1nvq', '1o3f', '1o5b', '1os0', '1oyt', '1p1q', '1ps3', '1q8t', '1q8u', '1qi0', '1r5y', '1sln', '1sqa', '1u1b', '1u33', '1uto', '1vso', '1w3k', '1w3l', '1w4o', '1xd0', '1yc1', '1z95', '1zea', '2brb', '2cbj', '2cet', '2d1o', '2d3u', '2fvd', '2fvd.zip', 'index.pkl']


In [51]:
# STEP 0 — IMPORTS & DEVICE SETUP
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import numpy as np
import pickle
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem, RDLogger

RDLogger.DisableLog('rdApp.*')   # suppress RDKit warnings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Step 0] Using device: {device}")
if device.type == "cuda":
    print(f"         GPU: {torch.cuda.get_device_name(0)}")
    print(f"         VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")



[Step 0] Using device: cuda
         GPU: Tesla T4
         VRAM: 15.6 GB


In [52]:
# STEP 1 — CONSTANTS & ATOM MAPPING
NUM_ATOM_TYPES = 10   # C, N, O, F, S, P, Cl, Br, I, other

# Maps atomic number → class index
ATOM_MAP = {
    6:  0,   # Carbon
    7:  1,   # Nitrogen
    8:  2,   # Oxygen
    9:  3,   # Fluorine
    16: 4,   # Sulfur
    15: 5,   # Phosphorus
    17: 6,   # Chlorine
    35: 7,   # Bromine
    53: 8,   # Iodine
    # anything else → 9 (other)
}

def atomic_num_to_idx(atom_nums: torch.Tensor) -> torch.Tensor:
    """Convert a tensor of atomic numbers to class indices."""
    return torch.tensor(
        [ATOM_MAP.get(a.item(), 9) for a in atom_nums],
        dtype=torch.long
    )

def atom_types_to_onehot(indices: torch.Tensor) -> torch.Tensor:
    """Convert class index tensor to one-hot float tensor."""
    return F.one_hot(indices, num_classes=NUM_ATOM_TYPES).float()

print("[Step 1] Constants & atom mapping defined.")



[Step 1] Constants & atom mapping defined.


In [53]:
# STEP 2 — DIFFUSION SCHEDULE (Variance-Preserving DDPM)

class DiffusionSchedule:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, device='cpu'):
        self.T      = T
        self.device = device

        # β schedule: linearly increases from beta_start → beta_end
        self.betas      = torch.linspace(beta_start, beta_end, T).to(device)
        self.alphas     = 1.0 - self.betas
        # ᾱₜ = cumulative product of αₛ for s=1..t
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)

    def add_noise(self, x0: torch.Tensor, t: torch.Tensor):
        """
        Forward process: q(xₜ | x₀) = √ᾱₜ·x₀ + √(1−ᾱₜ)·ε
        Args:
            x0 : clean coordinates,  shape (N, 3)
            t  : integer timestep,   shape (1,)
        Returns:
            xt  : noisy coordinates, shape (N, 3)
            eps : the noise added,   shape (N, 3)  ← model must predict this
        """
        alpha_bar_t = self.alpha_bars[t].view(1, 1)    # (1, 1) for broadcasting
        eps         = torch.randn_like(x0)
        xt          = torch.sqrt(alpha_bar_t) * x0 + torch.sqrt(1 - alpha_bar_t) * eps
        return xt, eps

    def get_alpha_bar(self, t: torch.Tensor) -> torch.Tensor:
        return self.alpha_bars[t]

schedule = DiffusionSchedule(T=1000, device=device)
print("[Step 2] Diffusion schedule (VP-DDPM) defined.")



[Step 2] Diffusion schedule (VP-DDPM) defined.


In [54]:
# STEP 3 — SINUSOIDAL TIMESTEP EMBEDDING
class SinusoidalTimestepEmbedding(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.linear1    = nn.Linear(hidden_dim, hidden_dim)
        self.linear2    = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            t : integer timestep tensor, shape (1,)
        Returns:
            embedding : shape (1, hidden_dim)
        """
        half  = self.hidden_dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device) / half
        ).float()
        # outer product: t (1,) × freqs (half,) → args (1, half)
        args      = t.float().unsqueeze(-1) * freqs.unsqueeze(0)
        embedding = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        embedding = torch.relu(self.linear1(embedding))
        embedding = self.linear2(embedding)
        return embedding   # (1, hidden_dim)

print("[Step 3] Sinusoidal timestep embedding defined.")

[Step 3] Sinusoidal timestep embedding defined.


In [55]:
# STEP 4 — LOCAL EQUIVARIANT LAYER  (intra-ligand / Local Branch)
class LocalEquivariantLayer(nn.Module):
    """Intra-ligand message passing — Local Branch of Dual EGNN."""

    def __init__(self, hidden_dim=64):
        super().__init__()
        # Message MLP: takes features of two atoms + their distance → message
        self.msg_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        # Feature update MLP: takes aggregated messages + t_emb → new features
        self.feat_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        # Coordinate update MLP: outputs a scalar weight per atom pair
        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, feat: torch.Tensor, coords: torch.Tensor,
                t_emb: torch.Tensor):
        """
        Args:
            feat   : atom features,     shape (N, H)
            coords : atom coordinates,  shape (N, 3)
            t_emb  : timestep embedding,shape (1, H)
        Returns:
            new_feat   : updated features,     shape (N, H)
            new_coords : updated coordinates,  shape (N, 3)
        """
        N = coords.size(0)

        # Pairwise relative positions and distances (equivariant)
        ci   = coords.unsqueeze(1).expand(N, N, 3)     # (N, N, 3)
        cj   = coords.unsqueeze(0).expand(N, N, 3)     # (N, N, 3)
        rel  = ci - cj                                  # (N, N, 3) relative pos
        dist = torch.norm(rel, dim=-1, keepdim=True)    # (N, N, 1) distances

        fi = feat.unsqueeze(1).expand(N, N, -1)
        fj = feat.unsqueeze(0).expand(N, N, -1)

        # Compute messages between all pairs
        msg_in = torch.cat([fi, fj, dist], dim=-1)     # (N, N, 2H+1)
        msgs   = self.msg_mlp(msg_in)                   # (N, N, H)

        # Mask self-interactions (i≠j)
        mask = (~torch.eye(N, dtype=torch.bool, device=coords.device)).unsqueeze(-1)
        msgs = msgs * mask                              # zero out diagonal

        agg = msgs.sum(dim=1)                           # (N, H) aggregated msgs

        # Update features: concat aggregated msgs + timestep info
        t_exp    = t_emb.expand(N, -1)                 # (N, H)
        new_feat = self.feat_mlp(
            torch.cat([feat + agg, t_exp], dim=-1)
        )                                               # (N, H)

        # Equivariant coordinate update: weighted sum of relative positions
        scalar     = self.coord_mlp(agg)               # (N, 1)
        coord_upd  = (scalar.unsqueeze(1) * rel).sum(dim=1)  # (N, 3)
        new_coords = coords + coord_upd

        return new_feat, new_coords

print("[Step 4] Local equivariant layer (intra-ligand) defined.")


[Step 4] Local equivariant layer (intra-ligand) defined.


In [56]:
# STEP 5 — GLOBAL EQUIVARIANT LAYER  (ligand ↔ pocket / Global Branch)
class GlobalEquivariantLayer(nn.Module):
    """Ligand-to-Pocket message passing — Global Branch of Dual EGNN."""

    def __init__(self, hidden_dim=64):
        super().__init__()
        self.msg_mlp = nn.Sequential(
            nn.Linear(hidden_dim + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.feat_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, lig_feat: torch.Tensor, lig_coords: torch.Tensor,
                poc_feat: torch.Tensor, poc_coords: torch.Tensor,
                t_emb: torch.Tensor):
        """
        Args:
            lig_feat   : ligand atom features,  shape (N, H)
            lig_coords : ligand atom coords,    shape (N, 3)
            poc_feat   : pocket atom features,  shape (M, H)
            poc_coords : pocket atom coords,    shape (M, 3)
            t_emb      : timestep embedding,    shape (1, H)
        Returns:
            new_lig_feat   : updated ligand features,    shape (N, H)
            new_lig_coords : updated ligand coordinates, shape (N, 3)
        """
        N = lig_coords.size(0)
        M = poc_coords.size(0)

        # Relative positions: each ligand atom i vs each pocket atom j
        li   = lig_coords.unsqueeze(1).expand(N, M, 3)  # (N, M, 3)
        pj   = poc_coords.unsqueeze(0).expand(N, M, 3)  # (N, M, 3)
        rel  = li - pj                                   # (N, M, 3)
        dist = torch.norm(rel, dim=-1, keepdim=True)     # (N, M, 1)

        # Pocket features expanded for each ligand atom
        pf     = poc_feat.unsqueeze(0).expand(N, M, -1) # (N, M, H)
        msg_in = torch.cat([pf, dist], dim=-1)           # (N, M, H+1)
        msgs   = self.msg_mlp(msg_in)                    # (N, M, H)
        agg    = msgs.mean(dim=1)                        # (N, H)

        # Update ligand features conditioned on pocket context + timestep
        t_exp        = t_emb.expand(N, -1)              # (N, H)
        new_lig_feat = self.feat_mlp(
            torch.cat([lig_feat + agg, t_exp], dim=-1)
        )                                                # (N, H)

        # Equivariant coordinate update toward/away from pocket
        scalar         = self.coord_mlp(agg)            # (N, 1)
        coord_upd      = (scalar.unsqueeze(1) * rel).mean(dim=1)  # (N, 3)
        new_lig_coords = lig_coords + coord_upd

        return new_lig_feat, new_lig_coords

print("[Step 5] Global equivariant layer (ligand↔pocket) defined.")



[Step 5] Global equivariant layer (ligand↔pocket) defined.


In [57]:
# STEP 6 — ATOM TYPE DIFFUSION UTILITIES
def add_type_noise(atom_types_onehot: torch.Tensor,
                   t: torch.Tensor, T: int = 1000) -> torch.Tensor:
    """
    Discrete atom type noising via linear interpolation to uniform.
    Args:
        atom_types_onehot : clean one-hot vectors, shape (N, NUM_ATOM_TYPES)
        t                 : integer timestep,       shape (1,)
        T                 : total timesteps
    Returns:
        noisy_types : soft distribution, shape (N, NUM_ATOM_TYPES)
    """
    alpha_t = 1.0 - (t.float().item() / T)     # signal strength: 1→0 as t→T
    uniform = torch.ones_like(atom_types_onehot) / NUM_ATOM_TYPES
    noisy   = alpha_t * atom_types_onehot + (1.0 - alpha_t) * uniform
    return noisy

print("[Step 6] Atom type diffusion utilities defined.")



[Step 6] Atom type diffusion utilities defined.


In [62]:
# STEP 7 — DATASET  (PDBBind Compatible)
class PDBBindDataset(Dataset):
    def __init__(self, base_path: str):
        self.base    = Path(base_path)
        self.samples = []

        for folder in sorted(self.base.iterdir()):
            if not folder.is_dir():
                continue
            pdb_id   = folder.name
            lig_path = folder / f"{pdb_id}_ligand.sdf"
            poc_path = folder / f"{pdb_id}_pocket.pdb"
            if lig_path.exists() and poc_path.exists():
                self.samples.append((lig_path, poc_path, pdb_id))

        print(f"[Step 7] Dataset loaded: {len(self.samples)} complexes from {base_path}")

    def __len__(self):
        return len(self.samples)

    def _load_ligand(self, lig_path):
        supplier = Chem.SDMolSupplier(str(lig_path), sanitize=False)
        mol      = supplier[0] if supplier else None
        if mol is None:
            return None, None
        try:
            conf = mol.GetConformer()
        except ValueError:
            return None, None

        coords = []
        atoms  = []
        for atom in mol.GetAtoms():
            pos = conf.GetAtomPosition(atom.GetIdx())
            coords.append([pos.x, pos.y, pos.z])
            atoms.append(atom.GetAtomicNum())

        return (torch.tensor(coords, dtype=torch.float32),
                torch.tensor(atoms,  dtype=torch.long))

    def _load_pocket(self, poc_path):
        mol = Chem.MolFromPDBFile(str(poc_path), sanitize=False)
        if mol is None:
            return None, None
        try:
            conf = mol.GetConformer()
        except ValueError:
            return None, None

        coords = []
        atoms  = []
        for atom in mol.GetAtoms():
            pos = conf.GetAtomPosition(atom.GetIdx())
            coords.append([pos.x, pos.y, pos.z])
            atoms.append(atom.GetAtomicNum())

        return (torch.tensor(coords, dtype=torch.float32),
                torch.tensor(atoms,  dtype=torch.long))

    def __getitem__(self, idx):
        lig_path, poc_path, pdb_id = self.samples[idx]

        lig_coords, lig_atoms = self._load_ligand(lig_path)
        poc_coords, poc_atoms = self._load_pocket(poc_path)

        if lig_coords is None or poc_coords is None:
            return self.__getitem__((idx + 1) % len(self))
        if lig_coords.size(0) < 3 or poc_coords.size(0) < 5:
            return self.__getitem__((idx + 1) % len(self))

          # ── CENTER both around ligand centroid ────────────────────────────────
          # This is critical — keeps coords in ~[-10, 10] range
        center     = lig_coords.mean(dim=0)      # (3,)
        lig_coords = lig_coords - center         # ligand centered at origin
        poc_coords = poc_coords - center         # pocket shifted by same amount
          # Both stay in correct relative geometry

        return {
              "lig_coords": lig_coords,
              "lig_atoms":  lig_atoms,
              "poc_coords": poc_coords,
              "poc_atoms":  poc_atoms,
              "pdb_id":     pdb_id
          }



In [63]:
# STEP 8 — FULL ImprovedPMDM MODEL
class ImprovedPMDM(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # ── Embeddings ──────────────────────────────────────────────────────
        self.atom_embed = nn.Embedding(100, hidden_dim)       # atomic number lookup
        self.type_embed = nn.Linear(NUM_ATOM_TYPES, hidden_dim)  # noisy type dist
        self.t_embed    = SinusoidalTimestepEmbedding(hidden_dim)  # timestep

        # ── Dual Branch Layers ───────────────────────────────────────────────
        self.local_layers  = nn.ModuleList([
            LocalEquivariantLayer(hidden_dim) for _ in range(num_layers)
        ])
        self.global_layers = nn.ModuleList([
            GlobalEquivariantLayer(hidden_dim) for _ in range(num_layers)
        ])

        # ── Fusion MLP (per layer) ───────────────────────────────────────────
        # Concatenates local + global features → projects back to hidden_dim
        self.fusion = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.SiLU(),
                nn.Linear(hidden_dim, hidden_dim)
            ) for _ in range(num_layers)
        ])

        # ── Output Heads ─────────────────────────────────────────────────────
        self.coord_head = nn.Linear(hidden_dim, 3)              # predict ε (noise)
        self.type_head  = nn.Linear(hidden_dim, NUM_ATOM_TYPES) # predict atom class

    def forward(self, lig_atoms, lig_coords, poc_atoms, poc_coords,
                noisy_types, t):
        """
        Args:
            lig_atoms   : (N,)  atomic numbers of ligand
            lig_coords  : (N,3) noisy ligand coordinates at timestep t
            poc_atoms   : (M,)  atomic numbers of pocket
            poc_coords  : (M,3) clean pocket coordinates (never noised)
            noisy_types : (N, NUM_ATOM_TYPES) soft corrupted atom type dist
            t           : (1,) integer timestep
        Returns:
            coord_noise_pred : (N,3) predicted noise ε on coordinates
            type_logits      : (N, NUM_ATOM_TYPES) atom type prediction
        """
        # Timestep embedding (shared across all layers)
        t_emb    = self.t_embed(t)                              # (1, H)

        # Pocket features (fixed — pocket is never updated)
        poc_feat = self.atom_embed(poc_atoms)                   # (M, H)

        # Ligand features: atomic identity + noisy type information
        lig_feat = (self.atom_embed(lig_atoms)
                    + self.type_embed(noisy_types))             # (N, H)

        # ── Run Dual Branches for num_layers iterations ───────────────────
        for local_layer, global_layer, fuse in zip(
            self.local_layers, self.global_layers, self.fusion
        ):
            # Local branch: ligand atom ↔ ligand atom
            local_feat, local_coords = local_layer(
                lig_feat, lig_coords, t_emb
            )
            # Global branch: ligand atom ↔ pocket atom
            global_feat, global_coords = global_layer(
                lig_feat, lig_coords, poc_feat, poc_coords, t_emb
            )

            # Fuse features: concat → MLP
            lig_feat = fuse(
                torch.cat([local_feat, global_feat], dim=-1)
            )                                                   # (N, H)
            # Fuse coordinates: simple average
            lig_coords = (local_coords + global_coords) / 2.0  # (N, 3)

        # ── Output Heads ──────────────────────────────────────────────────
        coord_noise_pred = self.coord_head(lig_feat)            # (N, 3)
        type_logits      = self.type_head(lig_feat)             # (N, NUM_ATOM_TYPES)

        return coord_noise_pred, type_logits

print("[Step 8] ImprovedPMDM model defined.")

# Quick sanity-check: count parameters
_tmp_model = ImprovedPMDM(hidden_dim=64, num_layers=3)
n_params   = sum(p.numel() for p in _tmp_model.parameters() if p.requires_grad)
print(f"         Trainable parameters: {n_params:,}")
del _tmp_model



[Step 8] ImprovedPMDM model defined.
         Trainable parameters: 215,955


In [64]:
# STEP 9 — TRAINING LOOP WITH HYBRID LOSS
def train(
    data_path   = "/content/mini_dataset",
    drive_path  = "/content/drive/MyDrive/pmdm_checkpoints",
    hidden_dim  = 64,
    num_layers  = 3,
    num_epochs  = 30,
    lr          = 1e-3,
    T           = 1000,
    save_every  = 5,
    resume_from = None      # set to checkpoint path to resume training
):
    # ── Setup ────────────────────────────────────────────────────────────────
    Path(drive_path).mkdir(parents=True, exist_ok=True)

    dataset  = PDBBindDataset(data_path)
    loader   = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=0)

    model     = ImprovedPMDM(hidden_dim=hidden_dim, num_layers=num_layers).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    sched_lr  = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    coord_loss_fn = nn.MSELoss()
    type_loss_fn  = nn.CrossEntropyLoss()

    start_epoch = 0

    # ── Resume from checkpoint ───────────────────────────────────────────────
    if resume_from and Path(resume_from).exists():
        ckpt        = torch.load(resume_from, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_epoch = ckpt["epoch"] + 1
        print(f"[Step 9] Resumed from epoch {start_epoch}")

    print(f"[Step 9] Starting training: {num_epochs} epochs, "
          f"{len(dataset)} complexes, device={device}")
    print(f"         hidden_dim={hidden_dim}, num_layers={num_layers}, lr={lr}")
    print("-" * 65)

    # ── Training Loop ────────────────────────────────────────────────────────
    for epoch in range(start_epoch, num_epochs):
        model.train()
        total_loss  = 0.0
        total_coord = 0.0
        total_type  = 0.0
        n_batches   = 0

        for batch in loader:
            # Unpack batch (batch_size=1 → index [0])
            lig_coords = batch["lig_coords"][0].to(device)   # (N, 3)
            lig_atoms  = batch["lig_atoms"][0].to(device)    # (N,)
            poc_coords = batch["poc_coords"][0].to(device)   # (M, 3)
            poc_atoms  = batch["poc_atoms"][0].to(device)    # (M,)

            # ── Sample random timestep ────────────────────────────────────
            t = torch.randint(0, T, (1,), device=device)     # (1,)

            # ── Coordinate diffusion ──────────────────────────────────────
            noisy_coords, true_noise = schedule.add_noise(lig_coords, t)

            # ── Atom type diffusion ───────────────────────────────────────
            atom_indices  = atomic_num_to_idx(lig_atoms).to(device)  # (N,)
            true_onehot   = atom_types_to_onehot(atom_indices)        # (N, 10)
            noisy_types   = add_type_noise(true_onehot, t, T=T)      # (N, 10)

            # ── Forward pass ──────────────────────────────────────────────
            coord_pred, type_logits = model(
                lig_atoms, noisy_coords,
                poc_atoms, poc_coords,
                noisy_types, t
            )

            # ── Compute losses ────────────────────────────────────────────
            loss_coord = coord_loss_fn(coord_pred, true_noise)
            loss_type  = type_loss_fn(type_logits, atom_indices)
            loss       = loss_coord + 0.5 * loss_type

            # ── Backprop ──────────────────────────────────────────────────
            optimizer.zero_grad()
            loss.backward()
            # Gradient clipping: prevents exploding gradients in GNNs
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss  += loss.item()
            total_coord += loss_coord.item()
            total_type  += loss_type.item()
            n_batches   += 1

        sched_lr.step()   # decay LR every 10 epochs

        avg_loss  = total_loss  / max(n_batches, 1)
        avg_coord = total_coord / max(n_batches, 1)
        avg_type  = total_type  / max(n_batches, 1)

        print(f"Epoch {epoch+1:03d}/{num_epochs} | "
              f"Loss: {avg_loss:.4f} | "
              f"Coord: {avg_coord:.4f} | "
              f"Type: {avg_type:.4f} | "
              f"LR: {sched_lr.get_last_lr()[0]:.6f}")

        # ── Save checkpoint every N epochs ───────────────────────────────
        if (epoch + 1) % save_every == 0:
            ckpt_path = Path(drive_path) / f"pmdm_epoch_{epoch+1:03d}.pt"
            torch.save({
                "epoch":           epoch,
                "model_state":     model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "loss":            avg_loss,
                "hidden_dim":      hidden_dim,
                "num_layers":      num_layers,
            }, ckpt_path)
            print(f"         ✓ Checkpoint saved → {ckpt_path}")

    print("-" * 65)
    print("[Step 9] Training complete.")
    return model



In [65]:
# STEP 10 — SAVE / LOAD CHECKPOINT UTILITIES

def save_checkpoint(model, optimizer, epoch, loss, path):
    """Save model checkpoint to Google Drive."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "epoch":           epoch,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "loss":            loss,
        "hidden_dim":      model.hidden_dim,
        "num_layers":      model.num_layers,
    }, path)
    print(f"[Step 10] Checkpoint saved → {path}")


def load_checkpoint(path, device=device):
    """Load checkpoint and reconstruct model."""
    ckpt       = torch.load(path, map_location=device)
    hidden_dim = ckpt.get("hidden_dim", 64)
    num_layers = ckpt.get("num_layers", 3)

    model = ImprovedPMDM(hidden_dim=hidden_dim, num_layers=num_layers).to(device)
    model.load_state_dict(ckpt["model_state"])

    print(f"[Step 10] Loaded checkpoint from epoch {ckpt['epoch']+1}, "
          f"loss={ckpt['loss']:.4f}")
    return model, ckpt

print("[Step 10] Checkpoint utilities defined.")


# =============================================================================
# ENTRY POINT — Run Training
# -----------------------------------------------------------------------------
# Edit the paths below to match your Colab setup, then run this cell.
#
# Recommended settings for T4 GPU:
#   hidden_dim = 64   →  fast, fits in T4 VRAM easily
#   num_layers = 3    →  good depth without being too slow
#   num_epochs = 30   →  enough to see loss decrease
#   save_every = 5    →  checkpoint every 5 epochs to Drive
#
# To resume after a Colab disconnect:
#   Set resume_from = "/content/drive/MyDrive/pmdm_checkpoints/pmdm_epoch_010.pt"
# =============================================================================

if __name__ == "__main__":
    trained_model = train(
        data_path   = "/content/mini_dataset",
        drive_path  = "/content/drive/MyDrive/pmdm_checkpoints",
        hidden_dim  = 64,
        num_layers  = 3,
        num_epochs  = 30,
        lr          = 1e-3,
        T           = 1000,
        save_every  = 5,
        resume_from = None,    # ← change to checkpoint path to resume
    )

[Step 10] Checkpoint utilities defined.
[Step 7] Dataset loaded: 50 complexes from /content/mini_dataset
[Step 9] Starting training: 30 epochs, 50 complexes, device=cuda
         hidden_dim=64, num_layers=3, lr=0.001
-----------------------------------------------------------------
Epoch 001/30 | Loss: 38554939.7065 | Coord: 38554456.8021 | Type: 965.6979 | LR: 0.001000
Epoch 002/30 | Loss: 668629.7629 | Coord: 667781.6504 | Type: 1696.2357 | LR: 0.001000
Epoch 003/30 | Loss: 11116.8661 | Coord: 11034.9739 | Type: 163.7837 | LR: 0.001000
Epoch 004/30 | Loss: 999.5972 | Coord: 975.3189 | Type: 48.5567 | LR: 0.001000
Epoch 005/30 | Loss: 19.2332 | Coord: 17.0452 | Type: 4.3760 | LR: 0.001000
         ✓ Checkpoint saved → /content/drive/MyDrive/pmdm_checkpoints/pmdm_epoch_005.pt
Epoch 006/30 | Loss: 1.4996 | Coord: 1.1333 | Type: 0.7324 | LR: 0.001000
Epoch 007/30 | Loss: 10.1323 | Coord: 9.5614 | Type: 1.1418 | LR: 0.001000
Epoch 008/30 | Loss: 7.0546 | Coord: 2.9005 | Type: 8.3083 | LR:

In [66]:
#Part 1 — Reverse Sampler
# =============================================================================
# REVERSE DIFFUSION SAMPLER
# -----------------------------------------------------------------------------
# This is what makes the model actually GENERATIVE.
# Given a protein pocket, it generates a brand new ligand molecule
# by starting from pure Gaussian noise and denoising step by step.
# =============================================================================

@torch.no_grad()
def generate_molecule(
    model,
    poc_coords,
    poc_atoms,
    num_atoms = 20,
    T         = 1000,
    device    = device
):
    model.eval()

    poc_coords = poc_coords.to(device)
    poc_atoms  = poc_atoms.to(device)

    # ── Start from pure noise ──────────────────────────────────────────────
    # Keep coords small — same scale as training data (centered, ~5Å range)
    xt = torch.randn(num_atoms, 3).to(device) * 2.0

    # Atom types: uniform
    ht = torch.ones(num_atoms, NUM_ATOM_TYPES).to(device) / NUM_ATOM_TYPES

    # Placeholder atom indices — updated each step from model prediction
    lig_atoms = torch.zeros(num_atoms, dtype=torch.long).to(device)

    # ── Reverse loop: t = T-1 down to 0 ───────────────────────────────────
    for t_val in reversed(range(0, T)):
        t = torch.tensor([t_val], device=device)

        alpha_bar_t  = schedule.alpha_bars[t_val]
        beta_t       = schedule.betas[t_val]
        alpha_t      = schedule.alphas[t_val]

        # ── Model predicts noise ───────────────────────────────────────────
        coord_noise_pred, type_logits = model(
            lig_atoms,
            xt,
            poc_atoms,
            poc_coords,
            ht,
            t
        )

        # ── Update atom types at EVERY step ───────────────────────────────
        # This is the key fix — sharpen type distribution continuously
        type_probs = torch.softmax(type_logits, dim=-1)  # (N, 10)
        # Blend: more confident as t decreases
        blend = 1.0 - (t_val / T)                        # 0→1 as t→0
        ht    = (1 - blend) * ht + blend * type_probs    # (N, 10)

        # Update atom index from current best guess each step
        lig_atoms = ht.argmax(dim=-1)                    # (N,)

        # ── DDPM coordinate update ─────────────────────────────────────────
        # x_{t-1} = (1/√αₜ) * (xₜ - βₜ/√(1-ᾱₜ) * ε_pred) + σₜ*z
        coeff1   = 1.0 / torch.sqrt(alpha_t)
        coeff2   = beta_t / torch.sqrt(1.0 - alpha_bar_t + 1e-8)
        mean_xt  = coeff1 * (xt - coeff2 * coord_noise_pred)

        if t_val > 0:
            sigma = torch.sqrt(beta_t)
            xt    = mean_xt + sigma * torch.randn_like(xt)
        else:
            xt    = mean_xt   # last step: pure prediction, no noise

    return xt.cpu(), ht.argmax(dim=-1).cpu()

In [67]:
#Part 2 — Convert Output to a Real Molecule
from rdkit import Chem
from rdkit.Chem import AllChem, rdDetermineBonds
import numpy as np

# Reverse mapping: class index → atomic number
IDX_TO_ATOMIC = {
    0: 6,   # Carbon
    1: 7,   # Nitrogen
    2: 8,   # Oxygen
    3: 9,   # Fluorine
    4: 16,  # Sulfur
    5: 15,  # Phosphorus
    6: 17,  # Chlorine
    7: 35,  # Bromine
    8: 53,  # Iodine
    9: 6,   # Other → default Carbon
}
def coords_to_molecule(coords, atom_types):
    if isinstance(coords, torch.Tensor):
        coords     = coords.numpy()
    if isinstance(atom_types, torch.Tensor):
        atom_types = atom_types.numpy()

    N   = len(atom_types)
    mol = Chem.RWMol()

    # ── Add atoms ──────────────────────────────────────────────────────────
    for idx in atom_types:
        atomic_num = IDX_TO_ATOMIC.get(int(idx), 6)
        mol.AddAtom(Chem.Atom(atomic_num))

    # ── Set 3D coordinates ─────────────────────────────────────────────────
    conf = Chem.Conformer(N)
    for i, (x, y, z) in enumerate(coords):
        conf.SetAtomPosition(i, (float(x), float(y), float(z)))
    mol.AddConformer(conf, assignId=True)

    # ── Bond thresholds per atom pair (in Ångströms) ──────────────────────
    # More nuanced than a fixed 1.8Å cutoff
    BOND_THRESHOLDS = {
        (6, 6):  1.65,   # C-C
        (6, 7):  1.55,   # C-N
        (6, 8):  1.55,   # C-O
        (6, 16): 1.85,   # C-S
        (7, 7):  1.50,   # N-N
        (7, 8):  1.50,   # N-O
        (8, 8):  1.55,   # O-O
    }

    def get_threshold(a1, a2):
        key = tuple(sorted([a1, a2]))
        return BOND_THRESHOLDS.get(key, 1.70)   # default 1.70Å

    # ── Add bonds between nearby atoms ────────────────────────────────────
    atomic_nums = [IDX_TO_ATOMIC.get(int(t), 6) for t in atom_types]
    bonds_added = set()

    for i in range(N):
        for j in range(i + 1, N):
            dist      = np.linalg.norm(coords[i] - coords[j])
            threshold = get_threshold(atomic_nums[i], atomic_nums[j])
            if dist < threshold and (i, j) not in bonds_added:
                mol.AddBond(i, j, Chem.BondType.SINGLE)
                bonds_added.add((i, j))

    # ── Sanitize ───────────────────────────────────────────────────────────
    try:
        Chem.SanitizeMol(mol)
        return mol.GetMol()
    except Exception:
        try:
            # Try removing Hs and sanitizing again
            mol = Chem.RemoveHs(mol, sanitize=False)
            Chem.SanitizeMol(mol)
            return mol.GetMol()
        except Exception:
            return None

In [68]:
#Part 3 — Evaluation Metrics
from rdkit.Chem import Descriptors, QED, Crippen

def evaluate_molecule(mol):
    """
    Compute standard drug-likeness metrics for a generated molecule.
    These are the same metrics used in the PMDM paper.
    """
    if mol is None:
        return None

    results = {}

    # ── Validity ───────────────────────────────────────────────────────────
    # Can RDKit parse this molecule without errors?
    try:
        Chem.SanitizeMol(mol)
        results["valid"] = True
    except:
        results["valid"] = False
        return results

    # ── SMILES ─────────────────────────────────────────────────────────────
    results["smiles"] = Chem.MolToSmiles(mol)

    # ── QED (Drug-likeness score, 0-1, higher is better) ──────────────────
    # Combines 8 molecular properties into one score
    # QED > 0.5 is considered drug-like
    results["QED"] = QED.qed(mol)

    # ── Lipinski's Rule of Five ────────────────────────────────────────────
    # Most oral drugs satisfy these rules
    mw    = Descriptors.MolWt(mol)
    logp  = Crippen.MolLogP(mol)
    hbd   = Descriptors.NumHDonors(mol)
    hba   = Descriptors.NumHAcceptors(mol)
    results["MW"]           = round(mw, 2)
    results["LogP"]         = round(logp, 2)
    results["HBD"]          = hbd
    results["HBA"]          = hba
    results["Lipinski"]     = (mw<=500 and logp<=5 and hbd<=5 and hba<=10)

    # ── SA Score (Synthetic Accessibility, 1-10, lower is easier to make) ─
    try:
        from rdkit.Chem import RDConfig
        import sys, os
        sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
        import sascorer
        results["SA_score"] = round(sascorer.calculateScore(mol), 2)
    except:
        results["SA_score"] = None

    return results


def batch_evaluate(mols):
    """Evaluate a list of generated molecules and print summary."""
    results  = [evaluate_molecule(m) for m in mols if m is not None]
    valid    = [r for r in results if r and r.get("valid")]

    print(f"Total generated  : {len(mols)}")
    print(f"Valid molecules  : {len(valid)} ({100*len(valid)/max(len(mols),1):.1f}%)")

    if valid:
        qeds      = [r["QED"] for r in valid]
        lipinski  = [r for r in valid if r["Lipinski"]]
        print(f"Avg QED          : {np.mean(qeds):.3f}  (target > 0.5)")
        print(f"Lipinski pass    : {len(lipinski)}/{len(valid)}")
        print(f"\nSample SMILES:")
        for r in valid[:3]:
            print(f"  {r['smiles']}")

In [72]:
#Part 4 — Putting It All Together
def generate_and_evaluate(
    model,
    dataset,
    n_molecules = 10,
    num_atoms   = 20
):
    """
    Generate molecules for pocket samples from the dataset
    and evaluate their quality.
    """
    print(f"Generating {n_molecules} molecules...")
    generated_mols = []

    for i in range(n_molecules):
        # Pick a random pocket from dataset
        sample     = dataset[i % len(dataset)]
        poc_coords = sample["poc_coords"]
        poc_atoms  = sample["poc_atoms"]

        # Generate molecule
        coords, atom_types = generate_molecule(
            model      = model,
            poc_coords = poc_coords,
            poc_atoms  = poc_atoms,
            num_atoms  = num_atoms
        )

        # Convert to RDKit mol
        mol = coords_to_molecule(coords, atom_types)
        generated_mols.append(mol)

        status = "✓" if mol is not None else "✗"
        print(f"  [{status}] Molecule {i+1}: "
              f"{'valid' if mol else 'invalid'}")

    print()
    batch_evaluate(generated_mols)
    return generated_mols


# ── Run generation ─────────────────────────────────────────────────────────
dataset = PDBBindDataset("/content/mini_dataset")

# Retrain with fixes applied
trained_model, _ = load_checkpoint(
    "/content/drive/MyDrive/pmdm_checkpoints/pmdm_epoch_030.pt"
)

# Then generate
mols = generate_and_evaluate(
    model       = trained_model,
    dataset     = dataset,
    n_molecules = 10,
    num_atoms   = 20
)

[Step 7] Dataset loaded: 50 complexes from /content/mini_dataset
[Step 10] Loaded checkpoint from epoch 30, loss=1.0169
Generating 10 molecules...
  [✓] Molecule 1: valid
  [✓] Molecule 2: valid
  [✓] Molecule 3: valid
  [✓] Molecule 4: valid
  [✓] Molecule 5: valid
  [✓] Molecule 6: valid
  [✓] Molecule 7: valid
  [✓] Molecule 8: valid
  [✓] Molecule 9: valid
  [✓] Molecule 10: valid

Total generated  : 10
Valid molecules  : 10 (100.0%)
Avg QED          : 0.416  (target > 0.5)
Lipinski pass    : 0/10

Sample SMILES:
  C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C
  C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C
  C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C.C
